# Exploratory Experiments: Global Lookahead Pruning

This notebook contains the original experimental workflow for Global Lookahead Pruning, including:

- GPT-2 loading
- WikiText-2 tokenization
- Global magnitude pruning
- Layerwise magnitude pruning
- Lookahead pruning
- Perplexity evaluation
- Sparsity calculation
- Fine-tuning experiments

The production-ready reusable code has been moved to the `src/` directory.

In [1]:
import torch
import math
import intel_extension_for_pytorch as ipex
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# For reproducibility
torch.manual_seed(42)
np.random.seed(42)
# Define cache directory
cache_dir = "C:/Users/SD/.cache/huggingface/hub"
# Use GPU if available
device = 'xpu'
print(f"Using device: {device}")

d:\Conda\envs\pytorch-arc\Lib\site-packages\torchvision\io\image.py:14: UserWarning: Failed to load image Python extension: 'Could not find module 'D:\Conda\envs\pytorch-arc\Lib\site-packages\torchvision\image.pyd' (or one of its dependencies). Try using the full path with constructor syntax.'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


Using device: xpu


In [2]:
# Load pre-trained GPT-2 model and tokenizer
from transformers import GPT2LMHeadModel, GPT2Tokenizer, AutoModel, AutoTokenizer, AutoModelForCausalLM

model = GPT2LMHeadModel.from_pretrained('gpt2', cache_dir=cache_dir)
tokenizer = GPT2Tokenizer.from_pretrained('gpt2', cache_dir=cache_dir)

# model = AutoModelForCausalLM.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B", cache_dir=cache_dir)
# tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B", cache_dir=cache_dir)

# model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", cache_dir=cache_dir)
# tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", cache_dir=cache_dir)

# model = AutoModelForCausalLM.from_pretrained("google/gemma-2b", cache_dir=cache_dir)
# tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b", cache_dir=cache_dir)

# model = AutoModelForCausalLM.from_pretrained("microsoft/phi-2", cache_dir=cache_dir)
# tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-2", cache_dir=cache_dir)

In [3]:
tokenizer.pad_token = tokenizer.eos_token
device = 'xpu'
torch.xpu.empty_cache()
print(f"Using device: {device}")
model.to(device)

Using device: xpu


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [5]:
  # Load dataset
# !pip install datasets

from datasets import load_dataset
dataset = load_dataset('wikitext', 'wikitext-2-v1', split='test', cache_dir=cache_dir)
# Remove empty texts
processed_dataset = [text for text in dataset["text"] if text.strip()]
processed_dataset[:10]

[' = Robert <unk> = \n',
 ' Robert <unk> is an English film , television and theatre actor . He had a guest @-@ starring role on the television series The Bill in 2000 . This was followed by a starring role in the play Herons written by Simon Stephens , which was performed in 2001 at the Royal Court Theatre . He had a guest role in the television series Judge John <unk> in 2002 . In 2004 <unk> landed a role as " Craig " in the episode " Teddy \'s Story " of the television series The Long Firm ; he starred alongside actors Mark Strong and Derek Jacobi . He was cast in the 2005 theatre productions of the Philip Ridley play Mercury Fur , which was performed at the Drum Theatre in Plymouth and the <unk> <unk> Factory in London . He was directed by John <unk> and starred alongside Ben <unk> , Shane <unk> , Harry Kent , Fraser <unk> , Sophie Stanton and Dominic Hall . \n',
 ' In 2006 , <unk> starred alongside <unk> in the play <unk> written by Mark <unk> . He appeared on a 2006 episode of th

TOKENZIATION

In [6]:
from torch.nn.utils.rnn import pad_sequence

def tokenize_dataset(dataset, tokenizer, max_length, batch_size):
    all_input_ids = []
    all_attention_masks = []
    all_labels = []

    for i in range(0, len(dataset), batch_size):
        batch = dataset[i:i + batch_size]
        encodings_batch = tokenizer(
            batch,
            truncation=True,
            padding='max_length',
            max_length=max_length,
            return_tensors='pt'
        )
        # Create labels as a copy of input_ids
        labels = encodings_batch['input_ids'].clone()
        # Replace padding token id with -100 so that these positions are ignored in the loss computation
        labels[labels == tokenizer.pad_token_id] = -100

        all_input_ids.append(encodings_batch['input_ids'])
        all_attention_masks.append(encodings_batch['attention_mask'])
        all_labels.append(labels)

    input_ids = torch.cat(all_input_ids, dim=0)
    attention_masks = torch.cat(all_attention_masks, dim=0)
    labels = torch.cat(all_labels, dim=0)

    return {
        'input_ids': input_ids,
        'attention_mask': attention_masks,
        'labels': labels
    }


In [7]:
# Tokenize the dataset
encodings = tokenize_dataset(processed_dataset, tokenizer, max_length=512, batch_size=8)
print("Tokenization completed! Shape:", encodings['input_ids'].shape)

Tokenization completed! Shape: torch.Size([2891, 512])


GLOBAL_MAGNITUDE PRUNING

In [8]:
import torch.nn.utils.prune as prune
from transformers.models.gpt2.modeling_gpt2 import Conv1D

In [9]:
import torch.nn.utils.prune as prune
from transformers.models.gpt2.modeling_gpt2 import Conv1D

def layerwise_magnitude_pruning(model, amount):
    # Iterate over all modules in the model
    for name, module in model.named_modules():
        if isinstance(module, (torch.nn.Linear, Conv1D)):
            try:
                # Apply L1 unstructured pruning for this module
                prune.l1_unstructured(module, name='weight', amount=amount)
                # Remove the pruning reparameterization to finalize the changes
                prune.remove(module, 'weight')
                print(f"Pruned module: {name}")
            except Exception as e:
                print(f"Skipping module {name} due to error: {e}")
    return model


In [10]:
import torch.nn.utils.prune as prune
from transformers.models.gpt2.modeling_gpt2 import Conv1D

def global_magnitude_pruning(model, amount):
    parameters_to_prune = []
    # Collect eligible layers (GPT-2 mainly uses Linear layers)
    for name, module in model.named_modules():
        if isinstance(module, (torch.nn.Linear, Conv1D)):
            parameters_to_prune.append((module, 'weight'))

    # Apply global unstructured pruning using L1 norm
    prune.global_unstructured(
        parameters_to_prune,
        pruning_method=prune.L1Unstructured,
        amount=amount
    )

    # **New step: Remove the pruning reparameterization**
    for module, _ in parameters_to_prune:
        prune.remove(module, 'weight')

    return model


PERPLEXITY


In [11]:
def evaluate_perplexity(model, encodings, batch_size):
    model.eval()  # Ensure model is in evaluation mode
    total_loss = 0.0
    total_tokens = 0

    input_ids = encodings['input_ids']
    attention_mask = encodings['attention_mask']
    labels = encodings['labels']
    num_samples = input_ids.size(0)

    with torch.no_grad():
        for i in range(0, num_samples, batch_size):
            batch_input_ids = input_ids[i:i+batch_size].to(device)
            batch_attention_mask = attention_mask[i:i+batch_size].to(device)
            batch_labels = labels[i:i+batch_size].to(device)

            outputs = model(
                input_ids=batch_input_ids,
                attention_mask=batch_attention_mask,
                labels=batch_labels
            )
            # Multiply batch loss by the number of non-ignored tokens
            non_ignored = torch.sum(batch_labels != -100).item()
            loss = outputs.loss * non_ignored
            total_loss += loss.item()
            total_tokens += non_ignored

    avg_loss = total_loss / total_tokens
    perplexity = np.exp(avg_loss)
    return perplexity


SPARSITY

In [12]:
def calculate_sparsity(model):
    """
    Calculates the fraction of weights that are exactly zero (sparsity) in the model.

    :param model: The GPT2 model.
    :return: Sparsity as a fraction.
    """
    total_params = 0
    zero_params = 0
    for name, module in model.named_modules():
        if hasattr(module, 'weight'):
            weight = module.weight
            total_params += weight.numel()
            zero_params += torch.sum(weight == 0).item()
    return zero_params / total_params

PRUNE MODEL AND EVALUTE

In [12]:
# Evaluate baseline metrics on the unpruned model
print("Evaluating baseline metrics...")
baseline_ppl = evaluate_perplexity(model, encodings, batch_size=2)
baseline_sparsity = calculate_sparsity(model)
print("Baseline perplexity:", baseline_ppl)
print("Baseline sparsity:", baseline_sparsity)

Evaluating baseline metrics...


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


KeyboardInterrupt: 

Global Magnitude Evaluation

In [14]:
# # Apply global magnitude pruning
print("Applying pruning...")
model_pruned = global_magnitude_pruning(model, amount=0.3)
print("Pruning completed!")


Applying pruning...


RuntimeError: Current platform can NOT allocate memory block with size larger than 4GB! Tried to allocate 5.75 GiB (GPU  0; 15.56 GiB total capacity; 5.75 GiB already allocated; 5.85 GiB reserved in total by PyTorch)

In [ ]:
# # Evaluate perplexity after pruning
print("Evaluating after pruning...")
ppl_pruned = evaluate_perplexity(model_pruned, encodings, batch_size=1)
print(f"Perplexity after pruning: {ppl_pruned:.2f}")

Evaluating after pruning...
Perplexity after pruning: 81.75


Layer_Magnitude Evaluation

In [ ]:
torch.xpu.empty_cache()

In [13]:
# # Apply global magnitude pruning
print("Applying pruning...")
model_pruned = layerwise_magnitude_pruning(model, amount=0.25)
print("Pruning completed!")

Applying pruning...
Pruned module: transformer.h.0.attn.c_attn
Pruned module: transformer.h.0.attn.c_proj
Pruned module: transformer.h.0.mlp.c_fc
Pruned module: transformer.h.0.mlp.c_proj
Pruned module: transformer.h.1.attn.c_attn
Pruned module: transformer.h.1.attn.c_proj
Pruned module: transformer.h.1.mlp.c_fc
Pruned module: transformer.h.1.mlp.c_proj
Pruned module: transformer.h.2.attn.c_attn
Pruned module: transformer.h.2.attn.c_proj
Pruned module: transformer.h.2.mlp.c_fc
Pruned module: transformer.h.2.mlp.c_proj
Pruned module: transformer.h.3.attn.c_attn
Pruned module: transformer.h.3.attn.c_proj
Pruned module: transformer.h.3.mlp.c_fc
Pruned module: transformer.h.3.mlp.c_proj
Pruned module: transformer.h.4.attn.c_attn
Pruned module: transformer.h.4.attn.c_proj
Pruned module: transformer.h.4.mlp.c_fc
Pruned module: transformer.h.4.mlp.c_proj
Pruned module: transformer.h.5.attn.c_attn
Pruned module: transformer.h.5.attn.c_proj
Pruned module: transformer.h.5.mlp.c_fc
Pruned module:

In [14]:
torch.xpu.empty_cache()

In [15]:
# # Evaluate perplexity after pruning
print("Evaluating after pruning...")
ppl_pruned = evaluate_perplexity(model_pruned, encodings, batch_size=16)
print(f"Perplexity after pruning: {ppl_pruned:.2f}")

Evaluating after pruning...


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Perplexity after pruning: 53.67


In [ ]:
# Text Generation with the pruned model (keep generation batch small for memory reasons)
input_text = "The future of AI is"
input_ids = tokenizer.encode(input_text, return_tensors="pt").to(device)
output_ids = model.generate(input_ids, max_length=50)
output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print("Generated text:", output_text)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Generated text: The future of AI is in the hands of the people. The current state of the field, and where it will take us in the near future.
This year, as a result of the pandemic and the general lack of focus on the development and use


In [16]:
pruned_sparsity = calculate_sparsity(model_pruned)
print("After prunig sparsity:", pruned_sparsity)

After prunig sparsity: 0.24876387546840753


In [12]:
torch.xpu.empty_cache()

LOOKAHEADPRUNING

In [ ]:
# # -------------------------------
# # NEW: Lookahead Pruning Function
# # -------------------------------
# def global_lookahead_pruning(model, calibration_input, prune_amount, prune_n=0, prune_m=0):
#     """
#     Applies look-ahead pruning on the GPT-2 model.

#     This method uses calibration_input to capture activations from all transformer layers,
#     then, for each hidden layer (except the last two), computes a new importance metric for each
#     weight as:

#       Importance = |W| * sqrt( |mean(Activation_{l+1}) - mean(Activation_{l+2})| )

#     We then prune (set to zero) the weights whose importance is below a threshold determined by prune_amount.

#     Args:
#       model: A GPT2LMHeadModel.
#       calibration_input: Tensor of input IDs to be used for calibration.
#       prune_amount: Fraction of weights to prune (e.g., 0.3 for 30%).
#       prune_n, prune_m: Optional parameters for structured pruning (if needed).

#     Returns:
#       The pruned model.
#     """
#     model.eval()
#     # Use the calibration data to compute activations for each layer.
#     # GPT-2 transformer layers are stored in model.transformer.h
#     # First, pass the calibration_input through the embedding layer.
#     calibration_input = calibration_input.to(model.device)
#     with torch.no_grad():
#         x = model.transformer.wte(calibration_input)
#         activations = []
#         # Iterate over transformer layers to capture activations.
#         for layer in model.transformer.h:
#             x = layer(x)[0]  # each layer returns a tuple; take the first element
#             activations.append(x)

#     # For each hidden layer (except the last two), apply lookahead pruning.
#     num_layers = len(model.transformer.h)
#     for i in range(num_layers - 2):
#         # Get the activations from the next two layers.
#         act_next = activations[i + 1]
#         act_next_next = activations[i + 2]

#         # Compute the average activation for each channel over batch and sequence dims.
#         # This gives a vector of shape (hidden_size,)
#         mean_act_next = act_next.mean(dim=(0, 1))
#         mean_act_next_next = act_next_next.mean(dim=(0, 1))

#         # Compute the absolute difference and then take its square-root.
#         # new_activation will be a vector of shape (hidden_size,)
#         new_activation = torch.abs(mean_act_next + mean_act_next_next)
#         new_activation = torch.sqrt(new_activation)

#         # For the i-th transformer layer, iterate over its Linear submodules.
#         current_layer = model.transformer.h[i]
#         for name, module in current_layer.named_modules():
#             if isinstance(module, torch.nn.Linear):
#                 # module.weight shape: (out_features, in_features)
#                 # Assume new_activation shape is (in_features,)
#                 importance = torch.abs(module.weight.data) * new_activation.unsqueeze(0)
#                 # Determine threshold for pruning based on prune_amount.
#                 threshold = torch.quantile(importance.flatten(), 1 - prune_amount)
#                 mask = importance > threshold
#                 # Update weights: zero out those below threshold.
#                 module.weight.data.mul_(mask.float())

#     return model

In [ ]:
# def global_lookahead_pruning(model, calibration_input, prune_amount, prune_n=0, prune_m=0):
#     """
#     Applies look-ahead pruning on the GPT-2 model.

#     This method uses calibration_input to capture activations from all transformer layers,
#     then, for each hidden layer (except the last two), computes a new importance metric for each
#     weight as:

#       Importance = |W| * sqrt( |mean(Activation_{l+1}) - mean(Activation_{l+2})| )

#     We then prune (set to zero) the weights whose importance is below a threshold determined by prune_amount.

#     Args:
#       model: A GPT2LMHeadModel.
#       calibration_input: Tensor of input IDs to be used for calibration.
#       prune_amount: Fraction of weights to prune (e.g., 0.3 for 30%).
#       prune_n, prune_m: Optional parameters for structured pruning (if needed).

#     Returns:
#       The pruned model and a list with pruning details for each pruned module.
#     """
#     import torch

#     model.eval()
#     calibration_input = calibration_input.to(model.device)
    
#     # Capture activations from all transformer layers.
#     with torch.no_grad():
#         x = model.transformer.wte(calibration_input)
#         activations = []
#         for layer in model.transformer.h:
#             x = layer(x)[0]  # each layer returns a tuple; take the first element
#             activations.append(x)
    
#     pruning_results = []
#     num_layers = len(model.transformer.h)
    
#     for i in range(num_layers - 2):
#         # Get activations from the next two layers.
#         act_next = activations[i + 1]
#         act_next_next = activations[i + 2]
        
#         # Compute the average activation for each channel over batch and sequence dimensions.
#         mean_act_next = act_next.mean(dim=(0, 1))
#         mean_act_next_next = act_next_next.mean(dim=(0, 1))
        
#         # Compute the absolute difference (using subtraction as in the docstring) and then its square-root.
#         new_activation = torch.abs(mean_act_next - mean_act_next_next)
#         new_activation = torch.sqrt(new_activation)
        
#         current_layer = model.transformer.h[i]
#         for name, module in current_layer.named_modules():
#             # Check for both nn.Linear and the GPT-2 Conv1D module.
#             if isinstance(module, torch.nn.Linear):
#                 # For nn.Linear, weight shape is (out_features, in_features)
#                 if module.weight.data.shape[1] != new_activation.shape[0]:
#                     print(f"Skipping module '{name}' at layer {i}: expected in_features {module.weight.data.shape[1]}, got {new_activation.shape[0]}")
#                     continue
#                 importance = torch.abs(module.weight.data) * new_activation.unsqueeze(0)
#             elif module.__class__.__name__ == "Conv1D":
#                 # For GPT-2 Conv1D, weight shape is typically (in_features, out_features)
#                 if module.weight.data.shape[0] != new_activation.shape[0]:
#                     print(f"Skipping Conv1D module '{name}' at layer {i}: expected in_features {module.weight.data.shape[0]}, got {new_activation.shape[0]}")
#                     continue
#                 # Transpose to treat it like (out_features, in_features)
#                 importance = torch.abs(module.weight.data).t() * new_activation.unsqueeze(0)
#             else:
#                 continue

#             # Determine threshold for pruning.
#             threshold = torch.quantile(importance.flatten(), 1 - prune_amount)
#             mask = importance > threshold
#             pruning_results.append({
#                 'layer_index': i,
#                 'module_name': name,
#                 'importance': importance,
#                 'threshold': threshold,
#                 'mask': mask
#             })
#             # Apply pruning: zero out weights below the threshold.
#             if isinstance(module, torch.nn.Linear):
#                 module.weight.data.mul_(mask.float())
#             elif module.__class__.__name__ == "Conv1D":
#                 # For Conv1D, transpose back after applying the mask.
#                 pruned_weight = (torch.abs(module.weight.data).t() * mask.float()).t()
#                 module.weight.data.copy_(pruned_weight)
    
#     return model, pruning_results


In [12]:
def global_lookahead_pruning_gpt2(model, calibration_input, prune_amount, scaling_factor=1.0, prune_n=0, prune_m=0, epsilon=1e-6):
    """
    Applies look-ahead pruning on the GPT-2 model.

    This method uses calibration_input to capture activations from all transformer layers,
    then, for each hidden layer (except the last two), computes a new importance metric for each
    weight as:

      Importance = |W| * (scaling_factor * sqrt( |mean(Activation_{l+1}) - mean(Activation_{l+2})| + epsilon ))

    We then prune (set to zero) the weights whose importance is below a threshold determined by prune_amount.
    The threshold is computed as the prune_amount quantile of nonzero importance values if available.

    Args:
      model: A GPT2LMHeadModel.
      calibration_input: Tensor of input IDs to be used for calibration.
      prune_amount: Fraction of weights to prune (e.g., 0.3 to prune the lowest 30% of nonzero importance values).
      scaling_factor: Multiplier to adjust the activation difference term.
      prune_n, prune_m: Optional parameters for structured pruning (if needed).
      epsilon: Small constant to avoid near-zero differences.

    Returns:
      The pruned model and a list with pruning details for each pruned module.
    """
    import torch

    model.eval()
    calibration_input = calibration_input.to(model.device)

    # Capture activations from all transformer layers.
    with torch.no_grad():
        x = model.transformer.wte(calibration_input)
        activations = []
        for layer in model.transformer.h:
            x = layer(x)[0]  # each layer returns a tuple; take the first element
            activations.append(x)

    pruning_results = []
    num_layers = len(model.transformer.h)

    for i in range(num_layers - 2):
        # Get activations from the next two layers.
        act_next = activations[i + 1]
        act_next_next = activations[i + 2]

        # Compute the average activation for each channel over batch and sequence dimensions.
        mean_act_next = act_next.mean(dim=(0, 1))
        mean_act_next_next = act_next_next.mean(dim=(0, 1))

        # Compute the activation difference term, scaled appropriately.
        new_activation = scaling_factor * torch.sqrt(torch.abs(mean_act_next - mean_act_next_next) + epsilon)

        current_layer = model.transformer.h[i]
        for name, module in current_layer.named_modules():
            # Process nn.Linear modules.
            if isinstance(module, torch.nn.Linear):
                if module.weight.data.shape[1] != new_activation.shape[0]:
                    print(f"Skipping module '{name}' at layer {i}: expected in_features {module.weight.data.shape[1]}, got {new_activation.shape[0]}")
                    continue
                importance = torch.abs(module.weight.data) * new_activation.unsqueeze(0)

            # Process GPT-2's Conv1D modules.
            elif module.__class__.__name__ == "Conv1D":
                if module.weight.data.shape[0] != new_activation.shape[0]:
                    print(f"Skipping Conv1D module '{name}' at layer {i}: expected in_features {module.weight.data.shape[0]}, got {new_activation.shape[0]}")
                    continue
                importance = torch.abs(module.weight.data).t() * new_activation.unsqueeze(0)
            else:
                continue

            # Flatten importance and consider only nonzero values if possible.
            imp_flat = importance.flatten()
            nonzero_imp = imp_flat[imp_flat > 0]
            if nonzero_imp.numel() > 0:
                threshold = torch.quantile(nonzero_imp, prune_amount)
            else:
                threshold = 0.0

            # Print diagnostic information.
            print(f"Layer {i}, Module {name}: importance stats -- min: {imp_flat.min().item():.6f}, max: {imp_flat.max().item():.6f}, mean: {imp_flat.mean().item():.6f}, std: {imp_flat.std().item():.6f}")
            print(f"Layer {i}, Module {name}: threshold = {threshold:.10f}")

            mask = importance > threshold

            num_weights = mask.numel()
            num_pruned = torch.sum(~mask).item()
            print(f"Layer {i}, Module {name}: pruned {num_pruned}/{num_weights} weights ({num_pruned/num_weights:.2%})")

            pruning_results.append({
                'layer_index': i,
                'module_name': name,
                'importance': importance,
                'threshold': threshold,
                'mask': mask
            })

            # Apply pruning: zero out weights below the threshold while preserving the original sign.
            if isinstance(module, torch.nn.Linear):
                module.weight.data.mul_(mask.float())
            elif module.__class__.__name__ == "Conv1D":
                pruned_weight = (module.weight.data.t() * mask.float()).t()
                module.weight.data.copy_(pruned_weight)

    return model, pruning_results


In [11]:
def global_lookahead_pruning_phi2(model, calibration_input, prune_amount, scaling_factor=1.0, prune_n=0, prune_m=0, epsilon=1e-6):
    """
    Applies look-ahead pruning on a Qwen2ForCausalLM model.
    
    This method uses calibration_input to capture activations from all transformer layers,
    then, for each hidden layer (except the last two), computes a new importance metric for each
    weight as:
    
      Importance = |W| * (scaling_factor * sqrt( |mean(Activation_{l+1}) - mean(Activation_{l+2})| + epsilon ))
    
    We then prune (set to zero) the weights whose importance is below a threshold determined by prune_amount.
    
    Args:
      model: A Qwen2ForCausalLM model.
      calibration_input: Tensor of input IDs for calibration.
      prune_amount: Fraction (e.g., 0.3) used to determine the pruning threshold on nonzero importance values.
      scaling_factor: Multiplier for the activation difference term.
      prune_n, prune_m: Optional parameters for structured pruning (unused here).
      epsilon: Small constant to avoid near-zero differences.
    
    Returns:
      The pruned model and a list with pruning details.
    """
    import torch

    model.eval()
    calibration_input = calibration_input.to(model.device)
    
    # Capture activations from all transformer layers.
    with torch.no_grad():
        # Get initial embeddings from the model's embed_tokens.
        x = model.model.embed_tokens(calibration_input)
        activations = []
        # Iterate over the Qwen2 transformer layers (model.model.layers is a ModuleList)
        for layer in model.model.layers:
            batch_size, seq_len, _ = x.size()
            # Create a dummy attention mask with the correct 4D shape: (batch_size, 1, 1, seq_len)
            attention_mask = torch.ones(batch_size, 1, 1, seq_len, device=x.device)
            # Generate position IDs (assume positions 0, 1, 2, ..., seq_len-1)
            position_ids = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size, -1)
            # Generate positional embeddings by passing both the hidden states and position IDs
            pos_emb = model.model.rotary_emb(x, position_ids)
            # Pass the inputs along with the required arguments.
            # The layer's forward method expects: hidden_states, attention_mask, position_ids, and position_embeddings.
            out = layer(x, attention_mask=attention_mask, position_ids=position_ids, position_embeddings=pos_emb)
            # Assuming the layer returns a tuple, take the first element as the output.
            x = out[0]
            activations.append(x)
    
    pruning_results = []
    num_layers = len(model.model.layers)
    
    # Iterate over layers except the last two for lookahead.
    for i in range(num_layers - 2):
        # Get activations from the next two layers.
        act_next = activations[i + 1]
        act_next_next = activations[i + 2]
        
        # Compute the average activation for each channel over batch and sequence dimensions.
        mean_act_next = act_next.mean(dim=(0, 1))
        mean_act_next_next = act_next_next.mean(dim=(0, 1))
        
        # Compute the activation difference term with scaling and epsilon.
        new_activation = scaling_factor * torch.sqrt(torch.abs(mean_act_next - mean_act_next_next) + epsilon)
        
        current_layer = model.model.layers[i]
        for name, module in current_layer.named_modules():
            # Process only Linear modules.
            if isinstance(module, torch.nn.Linear):
                if module.weight.data.shape[1] != new_activation.shape[0]:
                    print(f"Skipping module '{name}' at layer {i}: expected in_features {module.weight.data.shape[1]}, got {new_activation.shape[0]}")
                    continue
                importance = torch.abs(module.weight.data) * new_activation.unsqueeze(0)
            else:
                continue

            # Flatten importance and compute threshold based on nonzero values.
            imp_flat = importance.flatten()
            nonzero_imp = imp_flat[imp_flat > 0]
            if nonzero_imp.numel() > 0:
                # Move the tensor to CPU to compute the quantile safely
                nonzero_imp_cpu = nonzero_imp.detach().cpu()
                threshold = torch.quantile(nonzero_imp_cpu, prune_amount)
                # Optionally, move threshold back to the original device
                threshold = threshold.to(imp_flat.device)
            else:
                threshold = 0.0

            # Diagnostic prints for insight.
            print(f"Layer {i}, Module {name}: importance stats -- min: {imp_flat.min().item():.6f}, max: {imp_flat.max().item():.6f}, mean: {imp_flat.mean().item():.6f}, std: {imp_flat.std().item():.6f}")
            print(f"Layer {i}, Module {name}: threshold = {threshold:.10f}")

            mask = importance > threshold

            num_weights = mask.numel()
            num_pruned = torch.sum(~mask).item()
            print(f"Layer {i}, Module {name}: pruned {num_pruned}/{num_weights} weights ({num_pruned/num_weights:.2%})")

            pruning_results.append({
                'layer_index': i,
                'module_name': name,
                'importance': importance,
                'threshold': threshold,
                'mask': mask
            })

            # Apply pruning: zero out weights below the threshold.
            module.weight.data.mul_(mask.float())
    
    return model, pruning_results


In [14]:
# -------------------------------
# Evaluate baseline metrics on the unpruned model
print("Evaluating baseline metrics...")
baseline_ppl = evaluate_perplexity(model, encodings, batch_size=2)
baseline_sparsity = calculate_sparsity(model)
print("Baseline perplexity:", baseline_ppl)
print("Baseline sparsity:", baseline_sparsity)

Evaluating baseline metrics...


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Baseline perplexity: 45.81775430976414
Baseline sparsity: 0.0


In [18]:
torch.xpu.empty_cache()

In [12]:
import torch
from typing import Dict, Any, List
from transformers.models.gpt2.modeling_gpt2 import GPT2LMHeadModel, Conv1D

def global_lookahead_pruning_gpt2_report(
    model: GPT2LMHeadModel,
    calibration_input: torch.LongTensor,
    prune_amount: float,
    scaling_factor: float = 1.0,
    epsilon: float = 1e-6
) -> (GPT2LMHeadModel, Dict[str, Any]):
    """
    Applies look-ahead pruning on the GPT-2 model and returns a detailed pruning report.

    Returns a report containing for each pruned module:
      - theoretical_sparsity (mask-based)
      - actual_sparsity    (fraction of nonzero weights that were zeroed)
      - total_weights, pruned_weights
    plus overall and type-aggregated effective sparsities.
    """
    model.eval()
    device = next(model.parameters()).device
    calibration_input = calibration_input.to(device)

    # 1) capture activations
    with torch.no_grad():
        x = model.transformer.wte(calibration_input)
        activations = []
        for layer in model.transformer.h:
            x = layer(x)[0]
            activations.append(x)

    num_layers = len(model.transformer.h)
    modules_report: List[Dict[str, Any]] = []

    # global counters
    global_total = 0
    global_pruned = 0

    # type‐based counters
    type_totals = {"Linear": {"total": 0, "pruned": 0},
                   "Conv1D": {"total": 0, "pruned": 0}}

    # 2) for each layer except last two
    for i in range(num_layers - 2):
        act1, act2 = activations[i+1], activations[i+2]
        mean1 = act1.mean(dim=(0,1))
        mean2 = act2.mean(dim=(0,1))
        factor = scaling_factor * torch.sqrt(torch.abs(mean1 - mean2) + epsilon)

        layer = model.transformer.h[i]
        for name, module in layer.named_modules():
            if isinstance(module, torch.nn.Linear) or isinstance(module, Conv1D):
                W = module.weight.data
                total_w = W.numel()
                # count original zeros
                orig_zero = int((W == 0).sum().item())
                nonzero_before = total_w - orig_zero

                # build importance
                if isinstance(module, torch.nn.Linear):
                    if W.shape[1] != factor.shape[0]:
                        continue
                    imp = torch.abs(W) * factor.unsqueeze(0)
                    ptype = "Linear"
                else:
                    if W.shape[0] != factor.shape[0]:
                        continue
                    imp = torch.abs(W).t() * factor.unsqueeze(0)
                    ptype = "Conv1D"

                # compute threshold
                flat = imp.reshape(-1)
                nonzero_imp = flat[flat > 0]
                thresh = torch.quantile(nonzero_imp, prune_amount) if nonzero_imp.numel()>0 else 0.0

                # theoretical mask
                mask = imp > thresh
                theoretical_pruned = int((~mask).sum().item())
                theoretical_sparsity = theoretical_pruned / total_w

                # apply pruning
                if isinstance(module, torch.nn.Linear):
                    module.weight.data.mul_(mask.float())
                else:
                    module.weight.data.copy_((W.t()*mask.float()).t())

                # count new zeros
                new_zero = int((module.weight.data == 0).sum().item())
                actual_pruned = new_zero - orig_zero
                actual_sparsity = actual_pruned / nonzero_before if nonzero_before>0 else 0.0

                # accumulate globals
                global_total += nonzero_before
                global_pruned += actual_pruned
                type_totals[ptype]["total"]  += nonzero_before
                type_totals[ptype]["pruned"] += actual_pruned

                # record module report
                modules_report.append({
                    "layer": i,
                    "module": name,
                    "type": ptype,
                    "total_weights": total_w,
                    "orig_zeros": orig_zero,
                    "nonzero_before": nonzero_before,
                    "theoretical_pruned": theoretical_pruned,
                    "theoretical_sparsity": theoretical_sparsity,
                    "actual_pruned": actual_pruned,
                    "actual_sparsity": actual_sparsity,
                    "threshold": float(thresh),
                    "importance_stats": {
                        "min": float(flat.min().item()),
                        "max": float(flat.max().item()),
                        "mean": float(flat.mean().item()),
                        "std": float(flat.std().item())
                    }
                })

    overall_actual_sparsity = global_pruned / global_total if global_total>0 else 0.0
    per_type = {
        ptype: {
            "total_nonzero": vals["total"],
            "pruned": vals["pruned"],
            "actual_sparsity": vals["pruned"]/vals["total"] if vals["total"]>0 else 0.0
        }
        for ptype, vals in type_totals.items()
    }

    report = {
        "overall_actual_sparsity": overall_actual_sparsity,
        "by_param_type": per_type,
        "modules": modules_report
    }
    return model, report


In [12]:
import torch

def global_lookahead_pruning_gpt2_global(
    model: torch.nn.Module,
    calibration_input: torch.LongTensor,
    prune_amount: float,
    scaling_factor: float = 1.0,
    epsilon: float = 1e-6,
    calib_batch: int = 16,
    max_samples: int = 1_000_000,
):
    """
    Global Lookahead Pruning for GPT-2 using sampled‐quantiles:

    1) Stream per-layer mean activations (no large buffers).
    2) Compute each module's importance but don't prune yet.
    3) Randomly sample up to <max_samples> values from all importances.
    4) Compute quantile on that sample.
    5) Prune globally with that threshold.
    """
    model.eval()
    device = next(model.parameters()).device

    # 1) Stream forward for mean activations
    layers = model.transformer.h
    L = len(layers)
    D = model.config.n_embd

    running_sums = [torch.zeros(D, device=device) for _ in range(L)]
    total_tokens = 0

    for chunk in calibration_input.split(calib_batch, dim=0):
        with torch.no_grad():
            x = model.transformer.wte(chunk.to(device))  # [B,S,D]
            B, S, _ = x.size()
            total_tokens += B * S

            for i, block in enumerate(layers):
                x = block(x)[0]             # [B,S,D]
                running_sums[i] += x.sum((0,1))

    mean_acts = [running_sums[i] / total_tokens for i in range(L)]

    # 2) Compute and collect module importances (but do not prune yet)
    prunable = []
    all_sampled = []
    samples_needed = max_samples

    for i in range(L - 2):
        f_l = scaling_factor * torch.sqrt(
            torch.abs(mean_acts[i+1] - mean_acts[i+2]) + epsilon
        )  # [D]

        block = layers[i]
        for name, module in block.named_modules():
            if isinstance(module, torch.nn.Linear):
                W = module.weight.data          # [out, in]
                if W.size(1) != D:
                    continue
                imp = torch.abs(W) * f_l.unsqueeze(0)

            elif module.__class__.__name__ == "Conv1D":
                W = module.weight.data          # [in, out]
                if W.size(0) != D:
                    continue
                imp = torch.abs(W).t() * f_l.unsqueeze(0)

            else:
                continue

            flat = imp.flatten()
            prunable.append((module, isinstance(module, torch.nn.Linear), name, imp))

            # 3) Sample from this flat tensor for global quantile
            if samples_needed > 0:
                n = flat.numel()
                # pick min(n, samples_needed) random indices
                take = min(n, samples_needed)
                if take == n:
                    all_sampled.append(flat)
                else:
                    idx = torch.randperm(n, device=device)[:take]
                    all_sampled.append(flat[idx])
                samples_needed -= take

    # Concatenate sampled
    if len(all_sampled) == 0:
        raise RuntimeError("No nonzero importances found to sample from!")
    sampled = torch.cat(all_sampled)

    # 4) Compute quantile over the sample
    thresh = float(torch.quantile(sampled, prune_amount))

    # 5) Apply pruning globally
    total, pruned = 0, 0
    results = []
    for module, is_linear, name, imp in prunable:
        mask = imp > thresh
        num = mask.numel()
        pr = num - int(mask.sum().item())
        total += num
        pruned += pr

        # actually zero out
        if is_linear:
            module.weight.data.mul_(mask.float())
        else:  # Conv1D
            pruned_W = (module.weight.data.t() * mask.float()).t()
            module.weight.data.copy_(pruned_W)

        results.append({
            'module': name,
            'threshold': thresh,
            'pruned': pr,
            'total': num
        })

    sparsity = pruned / total
    print(f"Global threshold={thresh:.6f}, sparsity={sparsity:.2%}")

    return model, results


In [13]:
# For calibration data, take a small batch from the tokenized dataset.
calibration_input = encodings['input_ids'][:32]  # for example, use first 32 samples

In [ ]:
# Apply lookahead pruning (for example, prune 30% of weights in each eligible layer)
print("Applying lookahead pruning...")
model_pruned_lookahead, pr = global_lookahead_pruning_gpt2_report(model, calibration_input, prune_amount=0.9)
print("Lookahead pruning completed!")


Applying lookahead pruning...
Global threshold=0.251691, sparsity=93.55%
Lookahead pruning completed!


In [15]:
print(pr)

[{'module': 'attn.c_attn', 'threshold': 0.2516913115978241, 'pruned': 1592458, 'total': 1769472}, {'module': 'attn.c_proj', 'threshold': 0.2516913115978241, 'pruned': 561418, 'total': 589824}, {'module': 'mlp.c_fc', 'threshold': 0.2516913115978241, 'pruned': 2252589, 'total': 2359296}, {'module': 'attn.c_attn', 'threshold': 0.2516913115978241, 'pruned': 1705871, 'total': 1769472}, {'module': 'attn.c_proj', 'threshold': 0.2516913115978241, 'pruned': 585676, 'total': 589824}, {'module': 'mlp.c_fc', 'threshold': 0.2516913115978241, 'pruned': 2302895, 'total': 2359296}, {'module': 'attn.c_attn', 'threshold': 0.2516913115978241, 'pruned': 1690474, 'total': 1769472}, {'module': 'attn.c_proj', 'threshold': 0.2516913115978241, 'pruned': 587760, 'total': 589824}, {'module': 'mlp.c_fc', 'threshold': 0.2516913115978241, 'pruned': 2300118, 'total': 2359296}, {'module': 'attn.c_attn', 'threshold': 0.2516913115978241, 'pruned': 1698988, 'total': 1769472}, {'module': 'attn.c_proj', 'threshold': 0.251

In [16]:
print("Overall actual sparsity:", pr["overall_actual_sparsity"])
print("By type:", pr["by_param_type"])
for m in pr["modules"]:
    print(f"{m['layer']}.{m['module']} {m['type']}: "
          f"act_sparsity={m['actual_sparsity']:.2%}, "
          f"theo_sparsity={m['theoretical_sparsity']:.2%}")


Overall actual sparsity: 0.899999639723036
By type: {'Linear': {'total_nonzero': 0, 'pruned': 0, 'actual_sparsity': 0.0}, 'Conv1D': {'total_nonzero': 47185920, 'pruned': 42467311, 'actual_sparsity': 0.899999639723036}}
0.attn.c_attn Conv1D: act_sparsity=90.00%, theo_sparsity=90.00%
0.attn.c_proj Conv1D: act_sparsity=90.00%, theo_sparsity=90.00%
0.mlp.c_fc Conv1D: act_sparsity=90.00%, theo_sparsity=90.00%
1.attn.c_attn Conv1D: act_sparsity=90.00%, theo_sparsity=90.00%
1.attn.c_proj Conv1D: act_sparsity=90.00%, theo_sparsity=90.00%
1.mlp.c_fc Conv1D: act_sparsity=90.00%, theo_sparsity=90.00%
2.attn.c_attn Conv1D: act_sparsity=90.00%, theo_sparsity=90.00%
2.attn.c_proj Conv1D: act_sparsity=90.00%, theo_sparsity=90.00%
2.mlp.c_fc Conv1D: act_sparsity=90.00%, theo_sparsity=90.00%
3.attn.c_attn Conv1D: act_sparsity=90.00%, theo_sparsity=90.00%
3.attn.c_proj Conv1D: act_sparsity=90.00%, theo_sparsity=90.00%
3.mlp.c_fc Conv1D: act_sparsity=90.00%, theo_sparsity=90.00%
4.attn.c_attn Conv1D: act

In [16]:
torch.xpu.empty_cache()

In [17]:
# Evaluate perplexity after lookahead pruning
print("Evaluating after lookahead pruning...")
ppl_pruned_lookahead = evaluate_perplexity(model_pruned_lookahead, encodings, batch_size=16)
print(f"Perplexity after lookahead pruning: {ppl_pruned_lookahead:.2f}")
pruned_lookahead_sparsity = calculate_sparsity(model_pruned_lookahead)
print("Lookahead pruning sparsity:", pruned_lookahead_sparsity)

Evaluating after lookahead pruning...


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Perplexity after lookahead pruning: 371368.36
Lookahead pruning sparsity: 0.27091538443787166


In [23]:
# Text generation with the pruned model
input_text = "The future of AI is"
input_ids = tokenizer.encode(input_text, return_tensors="pt").to(device)
output_ids = model_pruned_lookahead.generate(input_ids, max_length=50)
output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print("Generated text:", output_text)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Generated text: The future of AI is a great deal of the same.







The first time you can be a newbie











The first time you can be a new



FINE TUNING

In [ ]:
# Function for fine-tuning the pruned model
def finetune_model(model, encodings, batch_size=1, epochs=3, lr=1e-5):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()

    for epoch in range(epochs):
        total_loss = 0
        for i in range(0, len(encodings['input_ids']), batch_size):
            input_ids = encodings['input_ids'][i:i+batch_size].to(device)
            attention_mask = encodings['attention_mask'][i:i+batch_size].to(device)
            labels = encodings['labels'][i:i+batch_size].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_loss += loss.item()
            loss.backward()
            optimizer.step()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss}")

# Function to test different prune amounts and fine-tune after pruning
def experiment_with_pruning_and_finetuning(model, encodings, calibration_input):
    prune_amounts = [0.3]  # Try different pruning amounts
    results = {}

    for prune_amount in prune_amounts:
        print(f"Testing prune_amount: {prune_amount}")
        # Apply lookahead pruning
        model_pruned_lookahead, pr = global_lookahead_pruning_qwen2(model, calibration_input, prune_amount=prune_amount)
        
        # Fine-tune the model after pruning
        finetune_model(model_pruned_lookahead, encodings, batch_size=1, epochs=3, lr=1e-5)

        # Evaluate perplexity after fine-tuning
        ppl_finetuned = evaluate_perplexity(model_pruned_lookahead, encodings, batch_size=1)
        print(f"Perplexity after fine-tuning (prune_amount={prune_amount}): {ppl_finetuned:.2f}")

        # Store results
        results[prune_amount] = {
            "perplexity": ppl_finetuned,
        }

    return results

# Example: Run the experiment
calibration_input = encodings['input_ids'][:32]  # Use the first 32 samples for calibration
prune_results = experiment_with_pruning_and_finetuning(model, encodings, calibration_input)

# Print out results
print("Results of different pruning amounts and fine-tuning:")
for prune_amount, metrics in prune_results.items():
    print(f"Prune Amount: {prune_amount:.3f} | Perplexity: {metrics['perplexity']:.3f} ")


Testing prune_amount: 0.3
Layer 0, Module self_attn.q_proj: importance stats -- min: 0.000000, max: 1.068338, mean: 0.021565, std: 0.028094
Layer 0, Module self_attn.q_proj: threshold = 0.0060554841
Layer 0, Module self_attn.q_proj: pruned 707790/2359296 weights (30.00%)
Layer 0, Module self_attn.k_proj: importance stats -- min: 0.000000, max: 0.428882, mean: 0.027502, std: 0.029132
Layer 0, Module self_attn.k_proj: threshold = 0.0090920273
Layer 0, Module self_attn.k_proj: pruned 117965/393216 weights (30.00%)
Layer 0, Module self_attn.v_proj: importance stats -- min: 0.000000, max: 0.226578, mean: 0.014481, std: 0.013976
Layer 0, Module self_attn.v_proj: threshold = 0.0053438577
Layer 0, Module self_attn.v_proj: pruned 117965/393216 weights (30.00%)
Layer 0, Module self_attn.o_proj: importance stats -- min: 0.000000, max: 0.569780, mean: 0.020461, std: 0.022464
Layer 0, Module self_attn.o_proj: threshold = 0.0069118976
Layer 0, Module self_attn.o_proj: pruned 707790/2359296 weights (

In [ ]:
# Updated fine-tune function
def finetune_model(model, encodings, batch_size=1, epochs=3, lr=1e-6):  # Lowered LR
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()

    for epoch in range(epochs):
        total_loss = 0
        for i in range(0, len(encodings['input_ids']), batch_size):
            input_ids = encodings['input_ids'][i:i+batch_size].to(device)
            attention_mask = encodings['attention_mask'][i:i+batch_size].to(device)
            labels = encodings['labels'][i:i+batch_size].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

            if torch.isnan(loss):
                print(f"⚠️ NaN detected at batch {i}. Skipping...")
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # ✅ Prevent exploding grads
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")


# experiment function
def experiment_with_pruning_and_finetuning(model, encodings, calibration_input):
    prune_amounts = [0.3]
    results = {}

    for prune_amount in prune_amounts:
        print(f"\n🔧 Testing prune_amount: {prune_amount}")

        # Apply pruning
        model_pruned_lookahead, pr = global_lookahead_pruning_qwen2(model, calibration_input, prune_amount=prune_amount)
        model_pruned_lookahead.to(dtype=torch.float32)

        # Garbage collect to avoid memory buildup
        gc.collect()
        torch.xpu.empty_cache()

        # Fine-tune pruned model
        finetune_model(model_pruned_lookahead, encodings, batch_size=1, epochs=3, lr=1e-6)

        # Evaluate perplexity after fine-tuning
        ppl_finetuned = evaluate_perplexity(model_pruned_lookahead, encodings, batch_size=1)
        print(f"✅ Perplexity after fine-tuning (prune_amount={prune_amount}): {ppl_finetuned:.2f}")

        results[prune_amount] = {
            "perplexity": ppl_finetuned,
        }

    return results


calibration_input = encodings['input_ids'][:32]
prune_results = experiment_with_pruning_and_finetuning(model, encodings, calibration_input)

print("\n📊 Results of different pruning amounts and fine-tuning:")
for prune_amount, metrics in prune_results.items():
    print(f"Prune Amount: {prune_amount:.3f} | Perplexity: {metrics['perplexity']:.3f}")



🔧 Testing prune_amount: 0.3
Layer 0, Module self_attn.q_proj: importance stats -- min: 0.000000, max: 48.798492, mean: 0.071748, std: 0.180266
Layer 0, Module self_attn.q_proj: threshold = 0.0183587112
Layer 0, Module self_attn.q_proj: pruned 707789/2359296 weights (30.00%)
Layer 0, Module self_attn.k_proj: importance stats -- min: 0.000000, max: 7.842101, mean: 0.090412, std: 0.142426
Layer 0, Module self_attn.k_proj: threshold = 0.0281882938
Layer 0, Module self_attn.k_proj: pruned 117965/393216 weights (30.00%)
Layer 0, Module self_attn.v_proj: importance stats -- min: 0.000000, max: 3.531054, mean: 0.031712, std: 0.052657
Layer 0, Module self_attn.v_proj: threshold = 0.0098229451
Layer 0, Module self_attn.v_proj: pruned 117965/393216 weights (30.00%)
Layer 0, Module self_attn.o_proj: importance stats -- min: 0.000000, max: 6.226357, mean: 0.037024, std: 0.065797
Layer 0, Module self_attn.o_proj: threshold = 0.0113876667
Layer 0, Module self_attn.o_proj: pruned 707793/2359296 weigh

In [15]:
torch.xpu.empty_cache()

In [17]:
print(model.dtype)

torch.float32


In [13]:
# For calibration data, take a small batch from the tokenized dataset.
calibration_input = encodings['input_ids'][:32]  # for example, use first 32 samples

In [14]:
import copy
import torch

def sensitivity_guided_lookahead_prune(
    model,
    calibration_input,
    total_prune=0.30,
    test_prune=0.05,
    eps: float = 1e-6
):
    """
    1) Measure per-layer sensitivity by pruning test_prune of that layer alone.
    2) Compute budget p_i for each layer so that sum(p_i)=total_prune.
    3) Apply lookahead pruning layerwise with prune_amount=p_i.
    """
    base_ppl = evaluate_perplexity(model, encodings, batch_size=2)
    layers = model.transformer.h[:-2]
    sensitivities = []
    prunable_layers = []

    # 1) Sensitivity measurement
    for i, block in enumerate(layers):
        m2 = copy.deepcopy(model).to(model.device)
        # prune only layer i by test_prune
        _ , _ = global_lookahead_pruning_gpt2(
            m2,
            calibration_input,
            prune_amount=test_prune,
            scaling_factor=1.0,
            epsilon=1e-6
        )
        ppl_i = evaluate_perplexity(m2, encodings, batch_size=2)
        sens = ppl_i - base_ppl
        sensitivities.append(sens)
        prunable_layers.append(i)
        print(f"Layer {i} sensitivity: ΔPPL={sens:.3f}")

    # 2) Inverse-sensitivity weights
    inv = [1.0/(s+eps) for s in sensitivities]
    s = sum(inv)
    alphas = [w/s for w in inv]

    # 3) Allocate and prune
    for idx, alpha in zip(prunable_layers, alphas):
        layer_prune = alpha * total_prune
        print(f"→ Pruning layer {idx} by {layer_prune:.1%}")
        _, _ = global_lookahead_pruning_gpt2(
            model,
            calibration_input,
            prune_amount=layer_prune,
            scaling_factor=1.0,
            epsilon=1e-6
        )

    return model


In [15]:

# 2) Run sensitivity-guided pruning
model_pruned = sensitivity_guided_lookahead_prune(
    model,
    calibration_input,
    total_prune=0.30,   # aim for 30% global
    test_prune=0.05     # small step to measure
)

print("Final sparsity:", calculate_sparsity(model_pruned))
print("Final PPL:", evaluate_perplexity(model_pruned, encodings, batch_size=2))


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Layer 0, Module attn.c_attn: importance stats -- min: 0.000000, max: 6.419120, mean: 0.100630, std: 0.133256
Layer 0, Module attn.c_attn: threshold = 0.0033928291
Layer 0, Module attn.c_attn: pruned 88474/1769472 weights (5.00%)
Layer 0, Module attn.c_proj: importance stats -- min: 0.000000, max: 8.448093, mean: 0.055144, std: 0.117939
Layer 0, Module attn.c_proj: threshold = 0.0010341378
Layer 0, Module attn.c_proj: pruned 29492/589824 weights (5.00%)
Layer 0, Module mlp.c_fc: importance stats -- min: 0.000000, max: 9.765187, mean: 0.078155, std: 0.089052
Layer 0, Module mlp.c_fc: threshold = 0.0036293513
Layer 0, Module mlp.c_fc: pruned 117965/2359296 weights (5.00%)
Skipping Conv1D module 'mlp.c_proj' at layer 0: expected in_features 3072, got 768
Layer 1, Module attn.c_attn: importance stats -- min: 0.000000, max: 4.360374, mean: 0.068979, std: 0.082818
Layer 1, Module attn.c_attn: threshold = 0.0031088935
Layer 1, Module attn.c_attn: pruned 88474/1769472 weights (5.00%)
Layer 1, M